# Clembench agents experiment constructor

**This is a ingle configurable notebook** for running ClemAgents on clembench games.

**Only Cell 2 (Config) needs to be changed between experiments.**

| Variable | Description |
|---|---|
| `GAME` | any clembench game (e.g., `taboo`, `referencegame`, etc.|
| `MODEL` | any model (e.g.,`qwen`, `gpt-4o-mini`) |
| `AGENT_TYPE` | a ClemAgent (choose from `agents.ipynb` or create a custom one) |
| `RUN_ID` | any string — used as the results folder name |
| `NUM_EPISODES` | integer |
| `SINGLE_PASS` | `True` = one pass through instances, `False` = cycle infinitely |


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
GAME         = ""   
MODEL        = ""          
AGENT_TYPE   = ""  
RUN_ID       = ""
NUM_EPISODES = 30
SINGLE_PASS  = True
# ───────────────────────────────────────────────────────────────────────

## 1. Preparation

In [ ]:
import os

CLEMBENCH_HOME = r"path-to-clembench"
#os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [ ]:
# uncomment and run once to install dependencies, then comment out again
# %pip install -r $CLEMBENCH_HOME/requirements.txt
# %pip install --upgrade ipywidgets jupyter_client clemcore

# Make tqdm usable in Jupyter notebooks
#%pip install --upgrade ipywidgets jupyter_client

In [ ]:
# Sanity check: version + confirm that the game is an available game
!clem --version
#!clem list games -s $GAME_NAME

In [ ]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model
from clemcore.clemgame import env, episode_results_folder_callbacks
from clemcore.backends import ModelRegistry, KeyRegistry

In [ ]:
#register the model if necessary

#registry = ModelRegistry.register("qwen3-vl:30b-a3b-thinking", backend="openai_compatible", model_id="qwen3-vl:30b-a3b-thinking")
#registry.get_first_model_spec_that_unify_with("qwen3-vl:30b-a3b-thinking")

In [ ]:
#register the API key and url if necessary

#API_KEY = "1234"
#ORGANIZATION = ""
#BASE_URL =""

#KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

## 2. Agent constructor

In [ ]:
%run ./agents.ipynb

## 3. Experiment setup

In [ ]:
def make_agent(agent_type: str) -> ClemAgent:
    """Instantiate an agent by name."""
    if agent_type == "BaselineAgentPlayer":
        return BaselineAgentPlayer()
    elif agent_type == "BasePlanningAgent":
        return BasePlanningAgent()
    elif agent_type == "FormatCheckingAgent":
        return FormatCheckingAgent()
    elif agent_type == "FormatCheckingAgent_LLM_judge":
        return FormatCheckingAgent_LLM_judge()
    elif agent_type == "UpfrontStrategyAgent":
        return UpfrontStrategyAgent()
    elif agent_type == "ReflexionAgent":
        return ReflexionAgent()
    else:
        raise ValueError(f"Unknown agent type: {agent_type!r}.")

In [ ]:
callbacks = episode_results_folder_callbacks(
    run_dir=RUN_ID,
    result_dir_path="playpen-records",
    player_model_infos=f"{AGENT_TYPE}-{MODEL}",
)


game_env = env(GAME, single_pass=SINGLE_PASS, callbacks=callbacks)

#removed reset

print("roles:", game_env.unwrapped.game_benchmark.game_spec["roles"])

In [ ]:
roles = game_env.unwrapped.game_benchmark.game_spec["roles"]
if GAME == "mm_mapworld":
      player_0 = make_agent(AGENT_TYPE)
      learner_agents = [player_0]
      agent_mapping = {"player_1": player_0}
elif len(roles) == 1 or GAME == "privateshared":
      player_0 = make_agent(AGENT_TYPE)
      learner_agents = [player_0]
      agent_mapping = {"player_0": player_0}
else:
      player_0 = make_agent(AGENT_TYPE)
      player_1 = make_agent(AGENT_TYPE)
      learner_agents = [player_0, player_1]
      agent_mapping = {"player_0": player_0, "player_1": player_1}

for agent_id, agent in agent_mapping.items():
      if isinstance(agent, ReflexionAgent):
          agent.agent_id = agent_id
          agent.memory = agent._load_memory()
          print(f"{agent_id}: {len(agent.memory)} memories loaded")  # <-- debug

print("Agent mapping:", {k: type(v).__name__ for k, v in agent_mapping.items()})

## 4. Run game

In [ ]:
import json as _json
from pathlib import Path

all_episodes_data = []
all_traces = []   #writing the traces across steps (for transcript injection)
results_folder = callbacks.callbacks[0].results_folder


def inject_traces_into_interactions(episode_path: Path, episode_traces: list):
      interactions_path = episode_path / "interactions.json"
      interactions = _json.load(open(interactions_path, encoding="utf-8"))

      player_names = [p for p in interactions["players"] if p != "GM"]
      agent_to_player = {f"player_{i}": name for i, name in enumerate(player_names)}

      already_injected = set() 

      for trace in episode_traces:
          player_name = agent_to_player.get(trace["agent"], "GM")
          context = trace["context"]

          # find the earliest turn not yet used for this player that matches context
          for turn_idx, turn in enumerate(interactions["turns"]):
              if (turn_idx, player_name) in already_injected:
                  continue
              for i, event in enumerate(turn):
                  match = event["action"].get("content") == context
                  if (event["from"] == "GM"
                          and event["to"] == player_name
                          and event["action"].get("content") == context):
                      
                      injected = [
                          {"from": player_name, "to": player_name,
                           "action": {"type": entry["type"], "content": entry["content"]}}
                          for entry in trace["trace"] if entry["type"] != "response"
                      ]
                      interactions["turns"][turn_idx] = (
                          turn[:i + 1] + injected + turn[i + 1:]
                      )
                      already_injected.add((turn_idx, player_name))
                      break
              else:
                  continue
              break 
      try:
          _json.dump(interactions, open(interactions_path, "w", encoding="utf-8"), indent=2)
      except Exception as e:
          print(f"Trace injection failed for episode {episode + 1}: {e}")
          import traceback
          traceback.print_exc()
          
        

for episode in range(NUM_EPISODES):
    game_env.reset()
    for agent in learner_agents:
        agent.reset()

    for agent_id, agent in agent_mapping.items():
          if isinstance(agent, ReflexionAgent):
              agent.agent_id = agent_id
              agent.memory = agent._load_memory()
              print(f"{agent_id}: {len(agent.memory)} memories loaded")

    episode_traces = []  # collects memory snapshots for this episode

    context_response_pairs = []
    for step_idx, agent_id in enumerate(game_env.agent_iter()):
        context, reward, termination, truncation, info = game_env.last()
        if termination or truncation:
          response = None
        elif agent_id in agent_mapping:
          response = agent_mapping[agent_id](context)
        else:
          response = game_env.unwrapped.player_by_agent_id[agent_id](context)


        agent = agent_mapping.get(agent_id)
        if agent is not None and hasattr(agent, 'trace') and agent.trace:
              episode_traces.append({
                  "step": step_idx,
                  "agent": agent_id,
                  "context": context["content"],
                  "trace": agent.trace
              })
              agent.trace = [] 
        
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

        print(f"  [obs:{agent_id}] {context['content'][:100]!r}")

    all_episodes_data.append(context_response_pairs)
    print(f"Episode {episode + 1}/{NUM_EPISODES} completed — {len(context_response_pairs)} steps")

    episode_path = results_folder.to_instance_dir_path(
          game_env.unwrapped.game_master,
          game_env.unwrapped.game_instance
      )
    _json.dump(episode_traces, open(episode_path / "agent_trace.json", "w"), indent=2)
    inject_traces_into_interactions(episode_path, episode_traces) 

    for agent_id, agent in agent_mapping.items():
          if isinstance(agent, ReflexionAgent):
              agent.reflect()

    print(f"Episode {episode + 1}/{NUM_EPISODES} done")

print(f"\nAll episodes done.")


print(f"\nMemory trace recorded.")

In [ ]:
# Display the last episode's steps
last_episode = all_episodes_data[-1]
print(f"Last episode: {len(last_episode)} steps")
print("-" * 60)
for idx, (agent_id, context, response, reward) in enumerate(last_episode):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"  Agent({agent_id}) <- Context: {context}")
    print(f"  Agent({agent_id}) -> Response: {response}")
    print("-" * 60)

## 5. After the game

In [ ]:
results_dir = callbacks.callbacks[0].results_folder.results_dir_path
run_dir     = callbacks.callbacks[0].results_folder.run_dir
print(f"Results saved to: {results_dir}")
print(f"Run dir:          {run_dir}")
print()
print("To score results:")
#do from clembench folder
print(f"  clem score -g {GAME} -r playpen-records") #or -r PATH_TO_FOLDER
print(f"  clem eval -r playpen-records")   #or -r PATH_TO_FOLDER

In [ ]:
# a small snippet to fix transcripts for (multimodal) referencegame
# (it's the only one that uses indexes, not turn names during scoring)
import glob
from pathlib import Path
import json

def repair_interactions(interactions_path, trace_path):
    interactions = json.load(open(interactions_path, encoding="utf-8"))
    traces = json.load(open(trace_path, encoding="utf-8"))

    player_names = [p for p in interactions["players"] if p != "GM"]

    #strip injected player-player events from all turns
    for turn_idx, turn in enumerate(interactions["turns"]):
        interactions["turns"][turn_idx] = [
            e for e in turn
            if not (e["from"] == e["to"] and e["from"] in player_names)
        ]

    #re-inject at end of turn
    agent_to_player = {f"player_{i}": name for i, name in enumerate(player_names)}
    already_injected = set()

    for trace in traces:
        player_name = agent_to_player.get(trace["agent"], "GM")
        context = trace["context"]
        for turn_idx, turn in enumerate(interactions["turns"]):
            if (turn_idx, player_name) in already_injected:
                continue
            for event in turn:
                if (event["from"] == "GM" and event["to"] == player_name
                        and event["action"].get("content") == context):
                    injected = [
                        {"from": player_name, "to": player_name,
                         "action": {"type": e["type"], "content": e["content"]}}
                        for e in trace["trace"]
                    ]
                    interactions["turns"][turn_idx] = turn + injected
                    already_injected.add((turn_idx, player_name))
                    break
            else:
                continue
            break

    json.dump(interactions, open(interactions_path, "w", encoding="utf-8"), indent=2)

#run on all referencegame episodes
base = r""
for interactions_path in glob.glob(
    base + "/referencegame/**/interactions.json",
    recursive=True
):
    trace_path = str(Path(interactions_path).parent / "agent_trace.json")
    if Path(trace_path).exists():
     #   repair_interactions(interactions_path, trace_path)
        print(f"Repaired: {interactions_path}")
    else:
        print(f"No trace file: {interactions_path}")